# EX_06 — Introducción a RAG (ejercicios)

**Notebook de referencia:** `notebook/06_Introduccion_RAG.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Plantilla de contexto

Escribe una función `build_prompt(context_chunks, question) -> str` que inserte los pasajes en un delimitador claro (`### Context` / `### Question`).


In [ ]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    """Build a grounded RAG prompt with explicit context delimiters."""
    separator = chr(10) + chr(10)
    context = separator.join(f"[{i+1}] {chunk}" for i, chunk in enumerate(context_chunks))
    return (
        "### Context" + chr(10)
        + context + chr(10) + chr(10)
        + "### Question" + chr(10)
        + question + chr(10) + chr(10)
        + "### Instructions" + chr(10)
        + "Answer only with information supported by the context. If the answer is not present, say so clearly."
    )

sample_chunks = [
    "TechCorp ofrece planes Basico, Profesional y Enterprise.",
    "El soporte atiende de lunes a viernes de 9:00 a 18:00.",
]
print(build_prompt(sample_chunks, "Cual es el horario de soporte?"))

## Actividad 2 — RAG sin LLM (retrieval only)

Con tus chunks del notebook teórico (o texto inventado), recupera top-k y **imprime** el contexto ensamblado sin llamar al generador.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = [
    "TechCorp ofrece planes Basico, Profesional y Enterprise.",
    "El soporte atiende de lunes a viernes de 9:00 a 18:00.",
    "La politica de devoluciones permite cancelar durante los primeros 30 dias.",
    "Las oficinas estan en Madrid, Barcelona y Valencia.",
]
question = "Cual es el horario de soporte?"

vectorizer = TfidfVectorizer().fit(chunks + [question])
chunk_vectors = vectorizer.transform(chunks)
query_vector = vectorizer.transform([question])
scores = cosine_similarity(query_vector, chunk_vectors)[0]

top_k = 2
top_indices = scores.argsort()[::-1][:top_k]
retrieved_context = [chunks[i] for i in top_indices]

print("Top-k chunks:")
for rank, idx in enumerate(top_indices, 1):
    print(f"{rank}. score={scores[idx]:.3f} | {chunks[idx]}")

print()
print("Prompt ensamblado:")
print(build_prompt(retrieved_context, question))

## Actividad 3 — Fallo de cobertura

Inventa un caso donde la respuesta **no** está en los chunks recuperados y describe en español (markdown) cómo lo detectarías en producción (p. ej. umbral de score, abstención).


_Tu explicacion:_

Un fallo de cobertura ocurre cuando el usuario pregunta algo que no esta en la base de conocimiento, por ejemplo: "Cual es la politica de vacaciones de TechCorp?" si solo tenemos documentos sobre precios, soporte y oficinas.

En produccion lo detectaria con una combinacion de senales:

1. Umbral minimo de similitud: si el mejor score esta por debajo de 0.25, el sistema se abstiene.
2. Verificacion de soporte: antes de generar, comprobar si algun chunk contiene terminos clave de la pregunta.
3. Respuesta con abstencion: si no hay evidencia, responder "No tengo informacion suficiente en la documentacion disponible".
4. Registro para mejora continua: guardar la pregunta sin respuesta para ampliar la base documental.